# RDA Monte Carlo — Análise e Visualizações

**Reestruturação Dinâmica de Ativos (RDA): Uma Abordagem Multicritérios de Recuperação Fiscal**

Este notebook reproduz os resultados da Seção 6 do artigo e gera os gráficos dos três cenários (pessimista, moderado, otimista).

Para executar: `pip install -r ../requirements.txt` e depois `Run All`.

In [ ]:
import sys
sys.path.insert(0, '../simulation')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from rda_montecarlo import run_simulation, summarize

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

results = run_simulation()
summary = summarize(results)
print('Simulation complete.')
summary

## 1. Distribuição do Impacto Fiscal Líquido Total

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(results['net_total_mln'], bins=80, color='steelblue', edgecolor='white', alpha=0.85)

p10 = results['net_total_mln'].quantile(0.10)
p50 = results['net_total_mln'].quantile(0.50)
p90 = results['net_total_mln'].quantile(0.90)

for val, label, color in [(p10,'P10 – Pessimista','tomato'), (p50,'P50 – Moderado','gold'), (p90,'P90 – Otimista','mediumseagreen')]:
    ax.axvline(val, color=color, linewidth=2, linestyle='--')
    ax.text(val + 5, ax.get_ylim()[1]*0.85, f'{label}\nR$ {val:.0f} mi', color=color, fontsize=9)

ax.set_xlabel('Impacto Fiscal Líquido Anual Médio (R$ milhões)', fontsize=11)
ax.set_ylabel('Frequência', fontsize=11)
ax.set_title('Distribuição do Impacto Fiscal Líquido — Modelo RDA\n(10.000 iterações Monte Carlo)', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/fig1_distribuicao_impacto.png', dpi=150)
plt.show()

## 2. Contribuição por Pilar (Box Plot)

In [ ]:
pillar_data = results[['pilar1_asset_mgmt_mln','pilar2_tech_schools_mln','pilar3_banrisul_mln']]
pillar_data.columns = ['Pilar 1\nGestão de Ativos','Pilar 2\nEscolas Técnicas','Pilar 3\nBanrisul']

fig, ax = plt.subplots(figsize=(9, 5))
pillar_data.boxplot(ax=ax, patch_artist=True,
    boxprops=dict(facecolor='steelblue', alpha=0.6),
    medianprops=dict(color='navy', linewidth=2))

ax.set_ylabel('Impacto Anual Médio (R$ milhões)', fontsize=11)
ax.set_title('Contribuição por Pilar — Distribuição Monte Carlo', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/fig2_contribuicao_pilares.png', dpi=150)
plt.show()

## 3. Tabela de Cenários (reproduz Tabela 3 do artigo)

In [ ]:
table3 = summary[summary['scenario_label'] != ''][[
    'scenario_label','pilar1_asset_mgmt_mln','pilar2_tech_schools_mln',
    'pilar3_banrisul_mln','net_total_mln','net_total_pct_rcl'
]].copy()

table3.columns = ['Cenário','Pilar 1 (R$ mi)','Pilar 2 (R$ mi)','Pilar 3 (R$ mi)','Total Líquido (R$ mi)','% RCL']
table3 = table3.reset_index(drop=True)

# Add implementation costs
table3['Custos Impl. (R$ mi)'] = -(table3['Total Líquido (R$ mi)'] * 0.15 / 0.85).round(1)

print('Tabela 3 — Impacto Fiscal Anual Médio do Modelo RDA (2025–2034)')
print(table3.to_string(index=False))
table3.to_csv('../outputs/tabela3_cenarios.csv', index=False)
table3

## 4. Análise de Sensibilidade — Variáveis Críticas

In [ ]:
import json
from pathlib import Path

# Pearson correlation between each input variable proxy and net total
corr = results.corr()['net_total_mln'].drop('net_total_mln').sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['tomato' if v < 0 else 'steelblue' for v in corr]
corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('Correlação de Pearson com Impacto Fiscal Líquido', fontsize=10)
ax.set_title('Análise de Sensibilidade — Variáveis por Importância Relativa', fontsize=11)
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('../outputs/fig3_sensibilidade.png', dpi=150)
plt.show()